# RAG Fine-tuning with Gemma 4 and BioASQ

Fine-tune Gemma 4 to answer questions based on provided context using the `rag-mini-bioasq` dataset.

In [1]:
# !pip install -q optuna evaluate tqdm rouge_score

In [ ]:
import pandas as pd
import optuna
import unsloth
import evaluate

from tqdm import tqdm
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments, Trainer

c:\Users\Acer\Documents\Academics\bemi\.venv\Lib\site-packages\unsloth\__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [ ]:
qa_dataset = load_dataset("rag-datasets/rag-mini-bioasq", name="question-answer-passages")
print("Available splits in qa_dataset:", qa_dataset.keys())
df = qa_dataset['test'].to_pandas()
print("Columns in df (from qa_dataset['test']):", df.columns)
print(f"Initial shape of df: {df.shape}")
print(f"NaNs in 'question' before dropna: {df['question'].isnull().sum()}")
print(f"NaNs in 'answer' before dropna: {df['answer'].isnull().sum()}")

text_corpus_dataset_dict = load_dataset("rag-datasets/rag-mini-bioasq", name="text-corpus")
text_corpus_df = text_corpus_dataset_dict['passages'].to_pandas() # Correctly get the 'passages' split as a DataFrame
print("Columns in text_corpus_df:", text_corpus_df.columns)

# Create a dictionary for quick lookup of passages by id
passage_lookup = text_corpus_df.set_index('id')['passage'].to_dict()

# Function to get passages from relevant_passage_ids, always returning a string
def get_passages_from_ids(id_list):
    if not isinstance(id_list, list) or not id_list: # Check if it's not a list or an empty list
        return "" # Return empty string instead of None
    found_passages = [passage_lookup.get(pid) for pid in id_list]
    return ' '.join(p for p in found_passages if p is not None and p != '')

# Apply this function to create the 'passages' column in df
df['passages'] = df['relevant_passage_ids'].apply(get_passages_from_ids)

# Clean NaNs - only for 'question' and 'answer', as 'passages' now always contains a string
df = df.dropna(subset=['question', 'answer'])
print(f"Shape of df after dropna: {df.shape}")

# Format: {query}\nContext:{relevant_passage}
def format_row(row):
    # Ensure passage is a string, handling cases where it might be empty or None after lookup/join
    passage = row['passages'] if isinstance(row['passages'], str) and row['passages'] else ""
    return f"{row['question']}\nContext:{passage}"

df['text'] = df.apply(format_row, axis=1)
df['output'] = df['answer']

Available splits in qa_dataset: dict_keys(['test'])
Columns in df (from qa_dataset['test']): Index(['question', 'answer', 'relevant_passage_ids', 'id'], dtype='object')
Initial shape of df: (4719, 4)
NaNs in 'question' before dropna: 0
NaNs in 'answer' before dropna: 0
Columns in text_corpus_df: Index(['passage', 'id'], dtype='object')
Shape of df after dropna: (4719, 5)


In [ ]:
df = df.drop(columns=['relevant_passage_ids', 'id', 'passages', 'question', 'output'])
df = df.rename(columns={'text': 'question'})

In [ ]:
print(df.head())

                                              answer  \
0  Coding sequence mutations in RET, GDNF, EDNRB,...   
1  The 7 known EGFR ligands  are: epidermal growt...   
2                Yes,  papilin is a secreted protein   
3  Long non coding RNAs appear to be spliced thro...   
4  Receptor activator of nuclear factor κB ligand...   

                                            question  
0  Is Hirschsprung disease a mendelian or a multi...  
1  List signaling molecules (ligands) that intera...  
2         Is the protein Papilin secreted?\nContext:  
3        Are long non coding RNAs spliced?\nContext:  
4        Is RANKL secreted from the cells?\nContext:  


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-E2B-it",
    max_seq_length=2048,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.5.6: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `llm_int8_enable_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
)

### Evaluation Setup & Hyperparameters
Evaluation on validation set and hyperparameter configuration.

In [ ]:
args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    max_steps=60,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    output_dir="outputs",
)

In [ ]:
def objective(trial):
    lr = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("per_device_train_batch_size", [2, 4])

    args = TrainingArguments(
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=20, # Short trial for demonstration
        learning_rate=lr,
        fp16=True,
        logging_steps=1,
        output_dir=f"outputs/trial_{trial.number}",
    )

    # trainer = Trainer(...) # Initialize trainer with args
    # return trainer.evaluate()["eval_loss"]
    return 0.5 # Placeholder loss

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=5)
print(study.best_params)

In [ ]:
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')

In [ ]:
val_df = qa_dataset['test'].to_pandas()
val_df['passages'] = val_df['relevant_passage_ids'].apply(get_passages_from_ids) # Reuse the function defined above
val_df = val_df.dropna(subset=['question', 'answer']) # Only drop NaNs for question/answer
val_df['text'] = val_df.apply(format_row, axis=1) # Reuse format_row defined above

In [ ]:
preds = []
for text_item in tqdm(val_df['text']):
	# Ensure text_item is a string. If it's None or NaN, convert to an empty string.
	if text_item is None or (isinstance(text_item, float) and pd.isna(text_item)):
		processed_text = ""
	else:
		processed_text = str(text_item) # Ensure it's a string

	inputs = tokenizer(processed_text, return_tensors="pt").to("cuda")
	outputs = model.generate(**inputs, max_new_tokens=100)
	preds.append(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
from unsloth import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=df, # Use the prepared DataFrame as the training set
    dataset_text_field="text",
    args=args,
    max_seq_length=2048,
)

In [ ]:
print(f"Number of None or NaN values in val_df['text']: {val_df['text'].isnull().sum()}")

In [ ]:
trainer.train()

After fine-tuning, you can optionally save the model.

In [ ]:
# Uncomment and run to save the model
# model.save_pretrained("fine_tuned_gemma4")
# tokenizer.save_pretrained("fine_tuned_gemma4")

In [ ]:
rouge_results = rouge.compute(predictions=preds, references=val_df['answer'])
print("ROUGE Metrics:", rouge_results)